# Stage 0 Kaggle Operations Console

This notebook is designed for Kaggle **Save & Run All**. The captured cell output is the debugging surface, so every code cell prints what it is doing, what a healthy result looks like, and what to fix when something fails.

## Setup checklist
- Accelerator: GPU T4 x2.
- Internet: ON.
- Kaggle secret: `HF_TOKEN` attached to this notebook, using a Hugging Face token with write access.
- `HF_USERNAME` below must match the Hub account or organization that owns the `stage0-*` repos.

## How to read this notebook's output
Look for lines starting with `CELL`, `EXPECTED`, `STATUS`, `SUCCESS`, `WARNING`, `SKIP`, or `ACTION`. A planned budget stop is success, not failure: run Save & Run All again next Kaggle session and the trainer resumes automatically from the Hub checkpoint.


In [ ]:
# CELL 1: User configuration. This is the only cell you should edit.
from pathlib import Path

VARIANT = "pdr"  # "pdr" | "gla" | "transformer"
HF_USERNAME = "your-username"
REPO_URL = "https://github.com/your-username/OMNI.git"  # If this is a placeholder, the notebook tries /kaggle/input fallback.
TOKENS = 2_500_000_000
MAX_HOURS = 8.5
HUB_KEEP_LAST = 2
EVAL_TOKENS = 1_000_000
RUN_FINAL_EVAL = True
COMPARE_ALL_VARIANTS = True

# Rescue knobs. Leave these alone unless the post-mortem tells you to change them.
SEQ_LEN = 1024
MICRO_BATCH = 2
CHUNK_LEN = 64
FORCE_FINAL_EVAL = False

WORK_DIR = Path("/kaggle/working")
INPUT_DIR = Path("/kaggle/input")
REPO_DIR = WORK_DIR / "OMNI"
TRAIN_DIR = REPO_DIR / "train"
OUTPUT_DIR = WORK_DIR / f"stage0-{VARIANT}"
CHECKPOINT_ROOT = OUTPUT_DIR / "checkpoints"
TRAIN_LOG_PATH = WORK_DIR / "train_log.txt"
HUB_REPO = f"{HF_USERNAME}/stage0-{VARIANT}"
HUB_URL = f"https://huggingface.co/{HUB_REPO}"

VALID_VARIANTS = {"pdr", "gla", "transformer"}
CONFIG_OK = VARIANT in VALID_VARIANTS and HF_USERNAME not in {"", "your-username"}

print("CELL: Configuration")
print("This cell defines the run. Later cells only read these values.")
print(f"variant={VARIANT}")
print(f"hub_repo={HUB_REPO}")
print(f"target_tokens={TOKENS:,} max_hours={MAX_HOURS} hub_keep_last={HUB_KEEP_LAST}")
print(f"seq_len={SEQ_LEN} micro_batch={MICRO_BATCH} chunk_len={CHUNK_LEN}")
print(f"output_dir={OUTPUT_DIR}")
if not CONFIG_OK:
    print("ACTION: Set VARIANT to pdr/gla/transformer and replace HF_USERNAME before training will run.")
else:
    print("STATUS: Config looks syntactically valid.")


In [ ]:
# CELL 2: Environment report. This catches the most common Kaggle setup mistakes early.
import os
import platform
import shutil
import sys

print("CELL: Environment report")
print("EXPECTED: a CUDA GPU. Any of L4 (~22GB), T4 x2, or P100 is fine; L4 is preferred.")
print("If you see CPU only, enable the GPU accelerator in Kaggle settings and rerun.")
print(f"python={sys.version.split()[0]} platform={platform.platform()}")

WORK_DIR.mkdir(parents=True, exist_ok=True)  # so the disk report below reflects /kaggle/working

ENV_OK = True
ENV_GPU_OK = False
try:
    import torch
    print(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()} cuda_version={torch.version.cuda}")
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    if gpu_count == 0:
        print("WARNING: No CUDA GPU detected. Training is skipped to avoid a slow CPU run.")
    else:
        ENV_GPU_OK = True
        for idx in range(gpu_count):
            props = torch.cuda.get_device_properties(idx)
            total_gb = props.total_memory / (1024 ** 3)
            print(f"gpu[{idx}]={props.name} memory={total_gb:.1f}GB")
except Exception as exc:
    ENV_OK = False
    print(f"WARNING: torch import or CUDA inspection failed: {exc}")
    print("ACTION: Restart the Kaggle session with a GPU accelerator and rerun all cells.")

disk = shutil.disk_usage(WORK_DIR)
print(f"disk_free={disk.free / (1024 ** 3):.1f}GB at {WORK_DIR}")
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"ram_available={ram.available / (1024 ** 3):.1f}GB ram_total={ram.total / (1024 ** 3):.1f}GB")
except Exception:
    try:
        pages = os.sysconf("SC_AVPHYS_PAGES")
        page_size = os.sysconf("SC_PAGE_SIZE")
        print(f"ram_available={pages * page_size / (1024 ** 3):.1f}GB")
    except Exception:
        print("ram_available=unknown")

if ENV_OK and ENV_GPU_OK:
    print("STATUS: Environment has CUDA. Continue to auth.")
else:
    print("ACTION: Fix the environment before training. Later cells will skip training if CUDA is missing.")

In [ ]:
# CELL 3: Hugging Face token, identity, write access, and private repo creation.
# Token sources are tried in order: (1) HF_TOKEN env var, (2) Kaggle secret HF_TOKEN.
# The env-var path makes this work on Kaggle images (e.g. newer L4 kernels) where the
# kaggle_secrets module is not injected, and on any non-Kaggle runtime.
import os

print("CELL: Hugging Face secret and repo auth")
print("This resolves an HF write token, verifies identity, and ensures the private checkpoint repo exists.")
AUTH_OK = False
hf_token = None
token_source = None

def _from_env():
    tok = os.environ.get("HF_TOKEN")
    return (tok, "env:HF_TOKEN") if tok else (None, None)

def _from_kaggle_secret():
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception as exc:
        print(f"STATUS: kaggle_secrets module not available ({exc}).")
        print("        This is normal on newer/L4 Kaggle kernels and off-Kaggle runtimes; using the env-var path instead.")
        return (None, None)
    try:
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        return (tok, "kaggle_secret:HF_TOKEN") if tok else (None, None)
    except Exception as exc:
        print(f"STATUS: kaggle_secrets is present but HF_TOKEN was not found ({exc}).")
        return (None, None)

if not CONFIG_OK:
    print("SKIP: Config is incomplete. Set HF_USERNAME and VARIANT in the config cell.")
else:
    for getter in (_from_env, _from_kaggle_secret):
        tok, source = getter()
        if tok:
            hf_token, token_source = tok, source
            os.environ["HF_TOKEN"] = tok  # export for the training subprocess
            print(f"STATUS: HF token loaded from {token_source}.")
            break

    if not hf_token:
        print("ACTION: No Hugging Face token found. Pick ONE fix, then Save & Run All again:")
        print("  A) SIMPLEST - switch the accelerator to 'GPU T4 x2' or 'GPU P100'. Those images include")
        print("     kaggle_secrets, so Add-ons -> Secrets -> HF_TOKEN (write token, attached to THIS notebook) just works.")
        print("  B) KEEP L4 - the L4 kernel lacks kaggle_secrets, so inject the token via the environment.")
        print("     Add a throwaway cell ABOVE this one:  import os; os.environ['HF_TOKEN'] = '<your hf_write_token>'")
        print("     then DELETE that cell before saving a public version.")
        print("  SECURITY: never commit a real token. If this repo is public, do not leave a token in any saved cell.")
    else:
        try:
            from huggingface_hub import HfApi, create_repo
            api = HfApi()
            who = api.whoami(token=hf_token)
            identity = who.get("name") or who.get("fullname") or who
            print(f"STATUS: whoami() succeeded for Hub identity: {identity}")
            create_repo(HUB_REPO, token=hf_token, private=True, exist_ok=True)
            AUTH_OK = True
            print("SUCCESS: Token can create or access the private repo, which confirms write access.")
            print(f"resolved_repo_url={HUB_URL}")
        except Exception as exc:
            message = str(exc)
            print(f"ACTION: Hub auth or repo creation failed: {message}")
            if any(code in message for code in ["401", "403", "Invalid user token", "Repository Not Found"]):
                print("Fix: create a Hugging Face token with WRITE access, refresh HF_TOKEN, and verify HF_USERNAME owns the namespace.")
            else:
                print("Fix: check Internet is ON and huggingface_hub installed correctly.")

In [ ]:
# CELL 4: Repo setup and dependency install. Torch is deliberately never installed or upgraded here.
import datetime as _dt
import shutil
import subprocess
import sys

print("CELL: Repo setup")
print("This cell clones or copies the OMNI repo, then installs train/requirements.txt with torch filtered out.")
SETUP_OK = False

def stamp():
    return _dt.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")

def run_command(cmd, *, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print(f"[{stamp()}] + {' '.join(cmd)}")
    proc = subprocess.run(cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if proc.stdout:
        print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
    print(f"[{stamp()}] returncode={proc.returncode}")
    return proc

def copy_repo_from_dataset():
    print("STATUS: Looking for a Kaggle dataset copy containing train/run_stage0.py.")
    if not INPUT_DIR.exists():
        return False
    matches = list(INPUT_DIR.rglob("train/run_stage0.py"))
    if not matches:
        return False
    source_root = matches[0].parents[1]
    print(f"STATUS: Found dataset repo candidate: {source_root}")
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.copytree(
        source_root,
        REPO_DIR,
        ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc", "train/runs", ".pytest_cache"),
    )
    return True

try:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    repo_url = REPO_URL.strip()
    placeholder_repo = (not repo_url) or "your-username" in repo_url

    if REPO_DIR.exists() and (REPO_DIR / "train" / "run_stage0.py").exists():
        print(f"STATUS: Reusing existing repo at {REPO_DIR}")
        if (REPO_DIR / ".git").exists() and not placeholder_repo:
            run_command(["git", "fetch", "--depth", "1", "origin"], cwd=REPO_DIR)
            run_command(["git", "pull", "--ff-only"], cwd=REPO_DIR)
    elif not placeholder_repo:
        proc = run_command(["git", "clone", "--depth", "1", repo_url, str(REPO_DIR)])
        if proc.returncode != 0:
            print("WARNING: git clone failed. Trying Kaggle dataset fallback next.")
            copied = copy_repo_from_dataset()
            if not copied:
                raise RuntimeError("git clone failed and no dataset fallback was found")
    else:
        print("STATUS: REPO_URL is a placeholder, so clone is skipped and dataset fallback is used.")
        copied = copy_repo_from_dataset()
        if not copied:
            raise RuntimeError("No repo found. Set REPO_URL or attach a Kaggle dataset containing this repo.")

    if not (TRAIN_DIR / "run_stage0.py").exists():
        raise RuntimeError(f"Repo setup did not produce {TRAIN_DIR / 'run_stage0.py'}")

    requirements = TRAIN_DIR / "requirements.txt"
    filtered = WORK_DIR / "requirements-no-torch.txt"
    removed = []
    kept = []
    for line in requirements.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if stripped.startswith("torch"):
            removed.append(line)
            continue
        kept.append(line)
    filtered.write_text("\n".join(kept) + "\n", encoding="utf-8")
    print(f"STATUS: Filtered requirements written to {filtered}")
    print(f"STATUS: Removed torch lines so Kaggle's CUDA-matched torch is preserved: {removed or 'none'}")
    proc = run_command([sys.executable, "-m", "pip", "install", "-r", str(filtered)])
    if proc.returncode != 0:
        raise RuntimeError("pip install failed")

    SETUP_OK = True
    print(f"SUCCESS: Repo is ready at {REPO_DIR}")
except Exception as exc:
    print(f"ACTION: Repo setup failed: {exc}")
    print("Most common fixes: set REPO_URL to a real Git URL, attach the repo as a Kaggle dataset, or rerun after a transient pip/network failure.")


In [ ]:
# CELL 5: Sanity tests and model parameter table. Do not train if this fails.
print("CELL: Sanity checks")
print("This runs pytest and prints the Stage 0 parameter table. Passing tests mean the repo and dependency set are coherent.")
SANITY_OK = False

if not SETUP_OK:
    print("SKIP: Repo setup failed, so pytest cannot run.")
else:
    pytest_proc = run_command([sys.executable, "-m", "pytest", "train/tests", "-q"], cwd=REPO_DIR)
    table_proc = run_command([sys.executable, "-c", "from perspective_torch import param_table; print(param_table(print_table=False))"], cwd=TRAIN_DIR)
    if pytest_proc.returncode == 0 and table_proc.returncode == 0:
        SANITY_OK = True
        print("SUCCESS: Sanity checks passed. Training can proceed if auth and GPU are also OK.")
    else:
        print("Do not proceed - the environment is broken; most common cause: pip resolved an incompatible package version. Full output above.")


In [ ]:
# CELL 6: Progress-so-far from cumulative Hub metrics.
import json
import math
import shutil
import statistics
import sys

print("CELL: Progress so far")
print("This downloads checkpoints/metrics.jsonl from the Hub if it exists, plots it, and estimates resume progress.")
PROGRESS_RECORDS = []
OBSERVED_TOKENS_PER_SEC = None
LATEST_STEP = 0
LATEST_TOKENS = 0

def read_jsonl(path):
    records = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            print(f"WARNING: Skipped malformed metrics line: {exc}")
    return records

if not SETUP_OK:
    print("SKIP: Repo setup failed, so progress cannot be inspected.")
else:
    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
    local_metrics = CHECKPOINT_ROOT / "metrics.jsonl"
    metrics_source = None
    latest_info = None

    if AUTH_OK:
        try:
            from huggingface_hub import hf_hub_download
            metrics_src = hf_hub_download(repo_id=HUB_REPO, filename="checkpoints/metrics.jsonl", token=hf_token)
            shutil.copy2(metrics_src, local_metrics)
            metrics_source = local_metrics
            print(f"STATUS: Downloaded cumulative Hub metrics to {local_metrics}")
        except Exception as exc:
            print(f"STATUS: No Hub metrics.jsonl found yet, or it could not be downloaded: {exc}")
        try:
            latest_src = hf_hub_download(repo_id=HUB_REPO, filename="checkpoints/latest.json", token=hf_token)
            latest_info = json.loads(Path(latest_src).read_text(encoding="utf-8"))
        except Exception as exc:
            print(f"STATUS: No Hub latest.json found yet: {exc}")
    elif local_metrics.exists():
        metrics_source = local_metrics
        print(f"STATUS: Auth unavailable; using local metrics at {local_metrics}")

    if metrics_source is None and local_metrics.exists():
        metrics_source = local_metrics

    if metrics_source is None:
        print("STATUS: Fresh start. No cumulative metrics found on the Hub or local disk.")
    else:
        PROGRESS_RECORDS = read_jsonl(metrics_source)
        if not PROGRESS_RECORDS:
            print("STATUS: metrics.jsonl exists but has no readable records yet.")
        else:
            try:
                sys.path.insert(0, str(TRAIN_DIR))
                from trainer import resolve_gradient_accumulation_steps
                grad_accum = resolve_gradient_accumulation_steps(seq_len=SEQ_LEN, micro_batch_size=MICRO_BATCH)
            except Exception:
                grad_accum = max(1, round(262_144 / (SEQ_LEN * MICRO_BATCH)))
            tokens_per_step = SEQ_LEN * MICRO_BATCH * grad_accum
            latest_record = max(PROGRESS_RECORDS, key=lambda item: int(item.get("step", 0)))
            LATEST_STEP = int((latest_info or {}).get("step", latest_record.get("step", 0)))
            LATEST_TOKENS = int(latest_record.get("tokens_total", LATEST_STEP * tokens_per_step))
            pct = 100.0 * LATEST_TOKENS / max(1, TOKENS)
            tps_values = [float(item["tokens_per_sec"]) for item in PROGRESS_RECORDS if item.get("tokens_per_sec")]
            if tps_values:
                OBSERVED_TOKENS_PER_SEC = statistics.median(tps_values[-10:])
                remaining_tokens = max(0, TOKENS - LATEST_TOKENS)
                remaining_hours = remaining_tokens / OBSERVED_TOKENS_PER_SEC / 3600.0
                sessions = math.ceil(remaining_hours / MAX_HOURS) if remaining_hours > 0 else 0
                speed_text = f"observed_tokens_per_sec={OBSERVED_TOKENS_PER_SEC:.1f} projected_sessions_remaining={sessions}"
            else:
                speed_text = "observed_tokens_per_sec=unknown projected_sessions_remaining=unknown"
            latest_name = (latest_info or {}).get("latest", "unknown")
            print(f"STATUS: Resume checkpoint={latest_name} step={LATEST_STEP} tokens={LATEST_TOKENS:,} pct_complete={pct:.2f}%")
            print(f"STATUS: {speed_text}")

            try:
                import matplotlib.pyplot as plt
                xs = [int(item.get("tokens_total", int(item.get("step", 0)) * tokens_per_step)) for item in PROGRESS_RECORDS]
                fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
                loss_points = [(x, item.get("loss")) for x, item in zip(xs, PROGRESS_RECORDS) if item.get("loss") is not None]
                eval_points = [(x, item.get("eval_ppl")) for x, item in zip(xs, PROGRESS_RECORDS) if item.get("eval_ppl") is not None]
                tps_points = [(x, item.get("tokens_per_sec")) for x, item in zip(xs, PROGRESS_RECORDS) if item.get("tokens_per_sec") is not None]
                if loss_points:
                    axes[0].plot([p[0] for p in loss_points], [p[1] for p in loss_points])
                axes[0].set_ylabel("train loss")
                if eval_points:
                    axes[1].plot([p[0] for p in eval_points], [p[1] for p in eval_points], marker="o")
                axes[1].set_ylabel("eval ppl")
                if tps_points:
                    axes[2].plot([p[0] for p in tps_points], [p[1] for p in tps_points])
                axes[2].set_ylabel("tokens/sec")
                axes[2].set_xlabel("tokens")
                fig.tight_layout()
                progress_png = WORK_DIR / f"progress_so_far_{VARIANT}.png"
                fig.savefig(progress_png, dpi=140)
                plt.close(fig)
                print(f"SUCCESS: Progress plot saved to {progress_png}")
            except Exception as exc:
                print(f"WARNING: Could not plot progress metrics: {exc}")


In [ ]:
# CELL 7: Training subprocess with live line-buffered output and a tee log.
import os
import subprocess
import sys

print("CELL: Training")
print("This launches train/run_stage0.py with PYTHONUNBUFFERED=1 and streams every line into both Kaggle output and train_log.txt.")
TRAIN_RETURN_CODE = None
TRAIN_CMD = []

ready = CONFIG_OK and ENV_OK and ENV_GPU_OK and AUTH_OK and SETUP_OK and SANITY_OK
if not ready:
    TRAIN_RETURN_CODE = -999
    print("SKIP: Training did not start because one or more prerequisites failed.")
    print(f"config_ok={CONFIG_OK} env_ok={ENV_OK} gpu_ok={ENV_GPU_OK} auth_ok={AUTH_OK} setup_ok={SETUP_OK} sanity_ok={SANITY_OK}")
else:
    TRAIN_CMD = [
        sys.executable,
        "train/run_stage0.py",
        "--variant", VARIANT,
        "--tokens", str(TOKENS),
        "--hub-repo", HUB_REPO,
        "--max-hours", str(MAX_HOURS),
        "--output-dir", str(OUTPUT_DIR),
        "--seq-len", str(SEQ_LEN),
        "--micro-batch", str(MICRO_BATCH),
        "--chunk-len", str(CHUNK_LEN),
        "--hub-keep-last", str(HUB_KEEP_LAST),
    ]
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["HF_TOKEN"] = hf_token
    print(f"[{stamp()}] + {' '.join(TRAIN_CMD)}")
    print(f"STATUS: Tee log path: {TRAIN_LOG_PATH}")
    try:
        with TRAIN_LOG_PATH.open("w", encoding="utf-8") as log_handle:
            log_handle.write(f"command={' '.join(TRAIN_CMD)}\n")
            process = subprocess.Popen(
                TRAIN_CMD,
                cwd=REPO_DIR,
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
            assert process.stdout is not None
            for line in iter(process.stdout.readline, ""):
                print(line, end="")
                log_handle.write(line)
                log_handle.flush()
            TRAIN_RETURN_CODE = process.wait()
        print(f"[{stamp()}] training_returncode={TRAIN_RETURN_CODE}")
    except Exception as exc:
        TRAIN_RETURN_CODE = -1
        with TRAIN_LOG_PATH.open("a", encoding="utf-8") as log_handle:
            log_handle.write(f"TRAINING_LAUNCH_EXCEPTION: {exc}\n")
        print(f"ACTION: Training subprocess failed to launch or stream: {exc}")


In [ ]:
# CELL 8: Post-mortem. This classifies the run outcome using return code and train_log.txt.
print("CELL: Post-mortem")
print("This cell is designed to run after the training cell even when training failed, because the training cell does not raise.")
TRAINING_COMPLETE = False
PLANNED_BUDGET_STOP = False
TRAIN_OUTCOME = "unknown"

log_text = ""
if TRAIN_LOG_PATH.exists():
    log_text = TRAIN_LOG_PATH.read_text(encoding="utf-8", errors="replace")
else:
    print(f"WARNING: No train log found at {TRAIN_LOG_PATH}")

def has_any(text, needles):
    lower = text.lower()
    return any(needle.lower() in lower for needle in needles)

if TRAIN_RETURN_CODE == -999:
    TRAIN_OUTCOME = "skipped"
    print("STATUS: Training was skipped because a prerequisite failed. Read the earlier ACTION lines.")
elif TRAIN_RETURN_CODE == 0 and "reason=budget" in log_text:
    TRAIN_OUTCOME = "budget"
    PLANNED_BUDGET_STOP = True
    print("SUCCESS (planned budget stop). This is normal. Run All again next session to continue.")
elif TRAIN_RETURN_CODE == 0 and "reason=complete" in log_text:
    TRAIN_OUTCOME = "complete"
    TRAINING_COMPLETE = True
    print("TRAINING COMPLETE.")
elif has_any(log_text, ["CUDA out of memory", "cuda runtime error", "outofmemoryerror"]):
    TRAIN_OUTCOME = "oom"
    print("ACTION: CUDA out of memory. Lower MICRO_BATCH to 1 in the config cell, or lower SEQ_LEN to 512.")
    print("Why: recurrent/attention temporaries include a (batch, d, chunk, chunk) style memory term, so batch, width, and chunk/sequence choices multiply quickly.")
elif has_any(log_text, ["401", "403", "Repository Not Found", "Invalid user token"]):
    TRAIN_OUTCOME = "auth"
    print("ACTION: Hugging Face auth failed. The token is missing write scope, expired, or HF_USERNAME points at the wrong namespace.")
    print("Fix: update the Kaggle HF_TOKEN secret with a write token and rerun from the auth cell.")
elif has_any(log_text, ["kaggle_secrets", "UserSecretsClient", "No secret"]):
    TRAIN_OUTCOME = "kaggle_secret"
    print("ACTION: Kaggle secret failed. Secrets are per-notebook; attach HF_TOKEN to THIS notebook, not only to another copy.")
elif has_any(log_text, ["ConnectionError", "ReadTimeout", "ChunkedEncodingError", "Temporary failure in name resolution", "NameResolutionError", "DNS"]):
    TRAIN_OUTCOME = "network"
    print("ACTION: Transient network failure to Hugging Face or the dataset stream. It is safe to Save & Run All again; resume is automatic.")
elif has_any(log_text, ["No space left", "disk full", "Errno 28"]):
    TRAIN_OUTCOME = "disk"
    print("ACTION: Disk is full. Kaggle /kaggle/working is small; delete old run dirs, then rerun.")
    print("Command: rm -rf /kaggle/working/stage0-* /kaggle/working/OMNI/train/runs")
elif TRAIN_RETURN_CODE == 0:
    TRAIN_OUTCOME = "zero_unclassified"
    print("STATUS: Training exited 0, but no budget or complete marker was found. Inspect the last log lines below.")
else:
    TRAIN_OUTCOME = "unrecognized_failure"
    print("ACTION: unrecognized failure - share these lines for debugging")

if TRAIN_OUTCOME in {"zero_unclassified", "unrecognized_failure"}:
    lines = log_text.splitlines()[-50:]
    print("--- last 50 train log lines ---")
    for line in lines:
        print(line)
    print("--- end log excerpt ---")


In [ ]:
# CELL 9: Final curves from cumulative metrics.jsonl.
import shutil

print("CELL: Curves")
print("This plots loss, eval perplexity, tokens/sec, and learning rate from cumulative metrics.jsonl.")
CURVE_FILES = []

def load_latest_metrics_for_curves():
    local_metrics = CHECKPOINT_ROOT / "metrics.jsonl"
    if local_metrics.exists():
        return local_metrics
    if AUTH_OK:
        try:
            from huggingface_hub import hf_hub_download
            src = hf_hub_download(repo_id=HUB_REPO, filename="checkpoints/metrics.jsonl", token=hf_token)
            CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, local_metrics)
            return local_metrics
        except Exception as exc:
            print(f"STATUS: Could not download metrics for final curves: {exc}")
    return None

metrics_path = load_latest_metrics_for_curves()
if metrics_path is None:
    print("SKIP: No metrics.jsonl is available yet, so there are no curves to plot.")
else:
    records = read_jsonl(metrics_path)
    if not records:
        print("SKIP: metrics.jsonl has no readable records.")
    else:
        try:
            import matplotlib.pyplot as plt
            plots_dir = CHECKPOINT_ROOT / "plots"
            plots_dir.mkdir(parents=True, exist_ok=True)
            fallback_tokens_per_step = max(1, SEQ_LEN * MICRO_BATCH * round(262_144 / max(1, SEQ_LEN * MICRO_BATCH)))
            xs = [int(item.get("tokens_total", int(item.get("step", 0)) * fallback_tokens_per_step)) for item in records]
            specs = [
                ("loss", "train loss", "stage0_loss.png"),
                ("eval_ppl", "eval perplexity", "stage0_eval_ppl.png"),
                ("tokens_per_sec", "tokens/sec", "stage0_tokens_per_sec.png"),
                ("lr", "learning rate", "stage0_lr.png"),
            ]
            for field, ylabel, filename in specs:
                points = [(x, item.get(field)) for x, item in zip(xs, records) if item.get(field) is not None]
                if not points:
                    print(f"STATUS: No {field} points yet; skipping {filename}")
                    continue
                fig, ax = plt.subplots(figsize=(9, 4))
                ax.plot([p[0] for p in points], [p[1] for p in points], marker="o" if field == "eval_ppl" else None)
                ax.set_xlabel("tokens")
                ax.set_ylabel(ylabel)
                ax.set_title(f"{VARIANT} {ylabel}")
                fig.tight_layout()
                work_file = WORK_DIR / f"{VARIANT}_{filename}"
                hub_file = plots_dir / f"{VARIANT}_{filename}"
                fig.savefig(work_file, dpi=150)
                fig.savefig(hub_file, dpi=150)
                plt.close(fig)
                CURVE_FILES.append(work_file)
                print(f"SUCCESS: Saved {work_file}")
            if AUTH_OK and CURVE_FILES:
                try:
                    from huggingface_hub import HfApi
                    HfApi().upload_folder(repo_id=HUB_REPO, folder_path=str(plots_dir), path_in_repo="checkpoints/plots", token=hf_token)
                    print("SUCCESS: Uploaded curve PNGs under checkpoints/plots so they are stored with checkpoint artifacts.")
                except Exception as exc:
                    print(f"WARNING: Curve PNG upload failed, but local PNGs were saved: {exc}")
        except Exception as exc:
            print(f"WARNING: Curve plotting failed: {exc}")


In [ ]:
# CELL 10: Final held-out evaluation on the latest checkpoint.
import sys

print("CELL: Final evaluation")
print("This runs train/eval_stage0.py for a larger held-out slice and prints variant, checkpoint step, loss, and perplexity.")
FINAL_EVAL_RETURN_CODE = None

if not RUN_FINAL_EVAL:
    print("SKIP: RUN_FINAL_EVAL is False.")
elif not (TRAINING_COMPLETE or FORCE_FINAL_EVAL):
    print("SKIP: Training is not complete. Set FORCE_FINAL_EVAL=True in the config cell only if you intentionally want an interim eval.")
elif not SETUP_OK:
    print("SKIP: Repo setup failed, so eval_stage0.py is unavailable.")
else:
    checkpoint_arg = str(CHECKPOINT_ROOT if CHECKPOINT_ROOT.exists() else HUB_REPO)
    cmd = [sys.executable, "train/eval_stage0.py", "--variant", VARIANT, "--checkpoint", checkpoint_arg, "--eval-tokens", str(EVAL_TOKENS)]
    proc = run_command(cmd, cwd=REPO_DIR)
    FINAL_EVAL_RETURN_CODE = proc.returncode
    if proc.returncode == 0:
        print("SUCCESS: Final eval completed. The report above is the held-out perplexity for this variant.")
    else:
        print("ACTION: Final eval failed. Read the subprocess output above; training checkpoints are still preserved.")


In [ ]:
# CELL 11: Cross-variant comparison and Stage 0 gate verdict.
import sys

print("CELL: Variant comparison")
print("This evaluates stage0-pdr, stage0-gla, and stage0-transformer repos and prints the full-rank vs low-rank gate verdict.")
COMPARE_RETURN_CODE = None

if not COMPARE_ALL_VARIANTS:
    print("SKIP: COMPARE_ALL_VARIANTS is False.")
elif not SETUP_OK:
    print("SKIP: Repo setup failed, so eval_stage0.py is unavailable.")
elif not AUTH_OK:
    print("SKIP: Hub auth failed, so private comparison repos cannot be downloaded.")
else:
    cmd = [sys.executable, "train/eval_stage0.py", "--compare", "--hub-user", HF_USERNAME, "--eval-tokens", str(EVAL_TOKENS)]
    proc = run_command(cmd, cwd=REPO_DIR)
    COMPARE_RETURN_CODE = proc.returncode
    print("Meaning: GATE PASS supports the full-rank PDR gate matching or beating the low-rank GLA gate at Stage 0; GATE FAIL means the low-rank baseline won this gate.")
    if proc.returncode != 0:
        print("ACTION: Comparison command failed before producing a verdict. Read the output above for the missing repo, auth, or eval error.")


## Final Next Actions

- If the post-mortem says **SUCCESS (planned budget stop)**: keep the same `VARIANT` and run Save & Run All again in the next Kaggle session. Resume is automatic from the Hub checkpoint and cumulative `metrics.jsonl`.
- If the post-mortem says **TRAINING COMPLETE** and final eval ran: keep the reported perplexity and run the comparison cell when all variants are complete.
- If the comparison prints **GATE PASS**: record that the full-rank PDR gate met the Stage 0 criterion against GLA, with transformer as reference.
- If the comparison prints **GATE FAIL**: switch project decisions to the GLA result or rerun the suspect variant only if the logs show an infrastructure failure.
- If any cell prints **ACTION**: make exactly that config, secret, GPU, network, or disk fix, then run Save & Run All again.
